# Flight Delay Domino Effect — Interactive Demo

Use the controls below to inspect a real aircraft-chain example from the April holdout and compare the saved model score under 0, 15, or 30 added minutes of scheduled turnaround slack.

**Project question:** can an inbound aircraft delay help identify whether its next flight will depart 15 or more minutes late?

This demo reuses the evaluated model outputs saved by Day 5. It does not retrain the model.

## What the interface shows

- April holdout performance: accuracy, recall, F1, and ROC-AUC.
- An example four-flight Southwest aircraft chain, selectable by flight route.
- Saved model score and simple residual-delay rule under the selected buffer.

**Important:** model scores are poorly calibrated. Treat them as relative scores, not literal delay probabilities. Buffer scenarios change one model input while holding everything else fixed. They are illustrative and do not estimate real operational savings.

In [1]:
from pathlib import Path
import html
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Find the project folder even if VS Code starts the notebook one directory above it.
ROOT = Path.cwd().resolve()
while not (ROOT / 'reports').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'reports').is_dir():
    raise FileNotFoundError('Open this notebook from the project folder, which contains reports/.')

REPORTS = ROOT / 'reports'
metrics = pd.read_csv(REPORTS / 'day3_april_final_metrics.csv')
selected = metrics.loc[metrics['MODEL'].str.contains('random forest', case=False)].iloc[0]
scenario_summary = pd.read_csv(REPORTS / 'day5_scenario_impact_summary.csv')
chain = pd.read_csv(REPORTS / 'day5_demo_chain_scenarios.csv', parse_dates=['FLIGHT_DATE'])

# Display only route/flight pairs present in the saved demo chain.
baseline = chain.loc[chain['EXTRA_SCHEDULED_SLACK_MIN'].eq(0)].copy()
baseline['ROUTE_LABEL'] = baseline.apply(lambda r: f"Flight {int(r['FLIGHT_NUM'])}  |  {r['ORIGIN']} → {r['DEST']}  |  {r['FLIGHT_DATE']:%b %d}", axis=1)
route_options = [(r['ROUTE_LABEL'], r['FLIGHT_NUM'], r['ORIGIN'], r['DEST']) for _, r in baseline.iterrows()]
print(f"Loaded {len(chain):,} saved scenario rows for {chain['TAIL_NUM'].nunique()} aircraft chain(s).")
print(f"April holdout: {int(selected['APRIL_TEST_ROWS']):,} linked flight pairs.")

ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
# Controls
route_picker = widgets.Dropdown(
    options=route_options,
    value=route_options[0][1:],
    description='Flight:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='520px'),
)
buffer_picker = widgets.ToggleButtons(
    options=[('Baseline', 0), ('+15 min', 15), ('+30 min', 30)],
    value=0,
    description='Extra slack:',
    style={'description_width': 'initial'},
)
output = widgets.Output()

def show_dashboard(change=None):
    flight_num, origin, dest = route_picker.value
    extra = buffer_picker.value
    match = chain.loc[
        chain['FLIGHT_NUM'].eq(flight_num)
        & chain['ORIGIN'].eq(origin)
        & chain['DEST'].eq(dest)
        & chain['EXTRA_SCHEDULED_SLACK_MIN'].eq(extra)
    ].iloc[0]
    aggregate = scenario_summary.loc[scenario_summary['EXTRA_SCHEDULED_SLACK_MIN'].eq(extra)].iloc[0]
    actual = '15+ min delay' if int(match['DEP_DEL15']) == 1 else 'Under 15 min delay'
    alert = 'Model flags this flight' if int(match['MODEL_ALERT_AT_LOCKED_THRESHOLD']) == 1 else 'Model does not flag this flight'
    score = float(match['MODEL_RELATIVE_SCORE'])
    residual = float(match['RULE_RESIDUAL_DELAY_MIN'])
    label = next(label for label, fn, org, dst in route_options if (fn, org, dst) == (flight_num, origin, dest))
    with output:
        output.clear_output(wait=True)
        display(widgets.HTML(f'''
        <div style="font-family:Segoe UI,Arial,sans-serif;color:#172b4d;max-width:950px">
          <div style="background:#102a43;color:white;padding:18px 22px;border-radius:10px 10px 0 0">
            <div style="font-size:13px;color:#9fd3ff;letter-spacing:1px">APRIL HOLDOUT · SOUTHWEST AIRCRAFT CHAIN</div>
            <div style="font-size:24px;font-weight:700;margin-top:6px">{html.escape(label)}</div>
            <div style="font-size:14px;margin-top:5px;color:#d9e2ec">Aircraft {html.escape(str(match['TAIL_NUM']))} · Previous arrival delay {float(match['PREV_ARR_DELAY_MIN']):.0f} min · Scheduled slack {float(match['SCHEDULED_TURNAROUND_SLACK_MIN']):.0f} min</div>
          </div>
          <div style="display:flex;flex-wrap:wrap;gap:24px;padding:20px 22px;background:#f0f6fa;border:1px solid #d9e2ec">
            <div><div style="font-size:13px;color:#52606d">MODEL RELATIVE SCORE</div><div style="font-size:28px;font-weight:700;color:#186faf">{score:.3f}</div></div>
            <div><div style="font-size:13px;color:#52606d">MODEL ALERT</div><div style="font-size:18px;font-weight:650">{html.escape(alert)}</div></div>
            <div><div style="font-size:13px;color:#52606d">SIMPLE RULE RESIDUAL</div><div style="font-size:22px;font-weight:700">{residual:.0f} min</div></div>
            <div><div style="font-size:13px;color:#52606d">OBSERVED OUTCOME</div><div style="font-size:18px;font-weight:650">{actual}</div></div>
          </div>
          <div style="padding:16px 22px;border:1px solid #d9e2ec;border-top:0">
            <b>All-April scenario context for +{extra} min:</b> mean relative score {float(aggregate['MEAN_MODEL_SCORE']):.3f}; simple rule flags {float(aggregate['SHARE_RULE_FLAGS_DELAYED'])*100:.2f}% of pairs.
            <div style="color:#7b341e;font-size:13px;margin-top:10px">Illustrative feature-only scenario. The observed outcome stays historical; this is not a causal estimate or a schedule recommendation.</div>
          </div>
        </div>
        '''))

route_picker.observe(show_dashboard, names='value')
buffer_picker.observe(show_dashboard, names='value')
display(widgets.VBox([route_picker, buffer_picker, output]))
show_dashboard()

## How to present this demo

1. Choose a route in the flight dropdown.
2. Switch between baseline, +15, and +30 minutes. Point out how the score and simple rule change.
3. Explain that the observed outcome is historical and cannot change. The scenario changes only scheduled slack.
4. Report the model’s April holdout metrics: 85.3% accuracy, 75.5% recall, 72.4% F1, and 0.887 ROC-AUC. Mention the calibration limitation.

If the dropdowns appear but the output does not render, use VS Code's **Trust Notebook** action and select the `.venv` Python kernel after installing `requirements.txt`.